# 🚀 Step 4: Export & Deploy 🟡

**ระดับ:** กลาง | **GPU:** T4 16GB | **เวลา:** ~20 นาที

## ขั้นตอน
1. โหลด Base Model + LoRA weights
2. **Merge** LoRA เข้ากับ Base Model
3. **Export** เป็น `.gguf` (16-bit และ 4-bit)
4. สร้าง **Modelfile** สำหรับ Ollama

## ทำไมต้อง Merge?

```
ก่อน Merge:                    หลัง Merge:
├── base_model/  (5GB)          └── merged_model/  (8GB)
└── lora_adapter/ (100MB)           → โหลดไฟล์เดียว
    → ต้องโหลดทั้งคู่               → ใช้กับ Ollama ได้
```

## ขนาดไฟล์ .gguf

| Format | ขนาด | คุณภาพ | เหมาะกับ |
|--------|------|--------|----------|
| f16 | ~16GB | สูงสุด | GPU ที่มี VRAM เยอะ |
| q8_0 | ~8GB | สูง | GPU/CPU RAM เยอะ |
| **q4_k_m** | **~4.5GB** | **ดี** | **CPU ทั่วไป (แนะนำ)** |
| q2_k | ~2.5GB | พอใช้ | CPU ที่ RAM น้อย |

---
> ⚡ **เปิด GPU ก่อน!** Runtime → Change runtime type → T4 GPU

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps peft accelerate bitsandbytes -q
print('✅ ติดตั้งเสร็จสิ้น')

## ⚙️ Config

เปลี่ยน `LORA_MODEL_PATH` ให้ตรงกับ step ที่ทำ:

In [ ]:
from unsloth import FastLanguageModel
import os

# 🔧 เปลี่ยนตาม step ที่ทำ:
# - จาก Step 1: "outputs/lora_model"
# - จาก Step 2: "outputs/lora_model_chat"
# - จาก Step 3: "outputs/lora_model_dpo"  ← แนะนำ
LORA_MODEL_PATH = "outputs/lora_model_dpo"

MAX_SEQ_LENGTH = 2048

print(f'📂 จะ export จาก: {LORA_MODEL_PATH}')

## 📥 โหลด LoRA Model

In [ ]:
print('📥 กำลังโหลดโมเดล...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print('✅ โหลดโมเดลสำเร็จ')

## 🔀 Merge LoRA เข้ากับ Base Model

```
W_merged = W_frozen + (lora_alpha/r) × A × B
```

**ข้อดี:** Inference เร็วขึ้น, ใช้กับ llama.cpp/Ollama ได้  
**ข้อเสีย:** ไม่สามารถ swap adapter ได้อีก, ไฟล์ใหญ่ขึ้น

In [ ]:
print('🔀 กำลัง Merge LoRA weights...')

# merge_and_unload(): merge LoRA เข้ากับ base model และ unload adapter
model = model.merge_and_unload()

print('✅ Merge สำเร็จ!')

## 💾 บันทึก Merged Model (HuggingFace format)

บันทึกในรูปแบบ HuggingFace ก่อน ใช้ได้กับ `transformers` library โดยตรง

In [ ]:
MERGED_DIR = "outputs/merged_model"
print(f'💾 บันทึก merged model ที่ {MERGED_DIR}...')

model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print('✅ บันทึก HuggingFace format สำเร็จ')

## 📦 Export เป็น .gguf (16-bit)

คุณภาพสูงสุด แต่ต้องการ disk space ~16GB

> ⚠️ ถ้า Colab disk ไม่พอ ให้ข้ามไป export แค่ q4_k_m

In [ ]:
# ตรวจสอบ disk space ก่อน
import shutil
total, used, free = shutil.disk_usage("/")
free_gb = free / 1024**3
print(f'💾 Disk ว่าง: {free_gb:.1f} GB')

if free_gb > 18:
    print('\n📦 กำลัง Export เป็น .gguf (16-bit)...')
    model.save_pretrained_gguf(
        "outputs/gguf_f16",
        tokenizer,
        quantization_method="f16",   # 16-bit float
    )
    print('✅ Export f16 สำเร็จ: outputs/gguf_f16/')
else:
    print('⚠️  Disk ไม่พอสำหรับ f16 (~16GB) → ข้ามไป q4_k_m')

## 📦 Export เป็น .gguf (4-bit q4_k_m) ← แนะนำ

`q4_k_m` คือ quantization method ที่นิยมที่สุดสำหรับ local deployment:
- `k` = k-quant (คุณภาพดีกว่า q4_0)
- `m` = medium (balance ระหว่าง size และ quality)
- ขนาด ~4.5GB สำหรับ 8B model
- รันได้บน CPU ที่มี RAM 8GB+

In [ ]:
print('📦 กำลัง Export เป็น .gguf (4-bit q4_k_m)...')

model.save_pretrained_gguf(
    "outputs/gguf_q4",
    tokenizer,
    quantization_method="q4_k_m",  # 4-bit quantized (แนะนำ)
)

print('✅ Export q4_k_m สำเร็จ: outputs/gguf_q4/')

## 📋 สร้าง Modelfile สำหรับ Ollama

Modelfile คือ config file สำหรับ Ollama คล้ายกับ Dockerfile แต่สำหรับ LLM

In [ ]:
modelfile_content = """# Ollama Modelfile
# สร้างโดย LLM Fine-tuning Course - Step 4

FROM ./model-q4_k_m.gguf

SYSTEM \"\"\"คุณคือผู้ช่วย AI ที่มีประโยชน์ ฉลาด และซื่อสัตย์
ตอบคำถามอย่างละเอียดและถูกต้อง ถ้าไม่รู้ให้บอกว่าไม่รู้\"\"\"

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER num_predict 512
PARAMETER repeat_penalty 1.1
"""

os.makedirs("outputs", exist_ok=True)
with open("outputs/Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile_content)

print('✅ สร้าง Modelfile สำเร็จ: outputs/Modelfile')
print('\n📄 เนื้อหา Modelfile:')
print(modelfile_content)

## 📤 Push ขึ้น HuggingFace Hub (Optional)

ถ้าต้องการ share โมเดลบน HuggingFace Hub

In [ ]:
# Uncomment และใส่ข้อมูลของคุณเพื่อ push ขึ้น Hub

# from huggingface_hub import login
# login(token="hf_your_token_here")  # ← ใส่ token ของคุณ

# # Push LoRA adapter (เล็ก ~100MB)
# model.push_to_hub("your-username/your-model-name", tokenizer=tokenizer)

# # หรือ Push .gguf ขึ้น Hub โดยตรง
# model.push_to_hub_gguf(
#     "your-username/your-model-name",
#     tokenizer,
#     quantization_method=["q4_k_m", "f16"],
# )

print('💡 Uncomment โค้ดด้านบนเพื่อ push ขึ้น HuggingFace Hub')

## 📊 สรุปไฟล์ที่ Export

In [ ]:
print('=' * 60)
print('🎉 Export เสร็จสมบูรณ์!')
print('=' * 60)
print('\n📁 ไฟล์ที่สร้าง:')

output_files = [
    ("outputs/merged_model/", "HuggingFace format"),
    ("outputs/gguf_f16/",     ".gguf 16-bit (คุณภาพสูงสุด)"),
    ("outputs/gguf_q4/",      ".gguf 4-bit q4_k_m (แนะนำ)"),
    ("outputs/Modelfile",     "Ollama Modelfile"),
]

for path, desc in output_files:
    exists = "✅" if os.path.exists(path) else "❌"
    print(f'  {exists} {path:<35} → {desc}')

print()
print('─' * 60)
print('🚀 วิธี Deploy บน Ollama:')
print('─' * 60)
print('''
# 1. ติดตั้ง Ollama
curl -fsSL https://ollama.com/install.sh | sh

# 2. Copy ไฟล์ไปยัง folder เดียวกัน
cp outputs/gguf_q4/*.gguf outputs/

# 3. สร้าง Ollama model
cd outputs && ollama create my-llama3 -f Modelfile

# 4. รัน!
ollama run my-llama3
''')
print('─' * 60)
print('🎓 จบคอร์ส LLM Fine-tuning แล้ว!')
print('   Step 1 → Step 2 → Step 3 → Step 4 ✅')